# SlidyReplay (playbook v0.0.3)

Generates MP4 videos of sliding puzzle replays.

In [ ]:
#@title ## Setup
import os, subprocess, sys

def run(cmd, show=False):
    r = subprocess.run(cmd, shell=True, capture_output=not show, text=True)
    if r.returncode != 0 and r.stderr:
        sys.stderr.write(r.stderr)
    return r.returncode == 0

if not os.path.exists("slidyreplay"):
    ok = run("git clone https://github.com/dphdmn/slidyreplay.git")
    print(f"{'  OK' if ok else 'FAIL'} git clone")
if os.path.basename(os.getcwd()) != "slidyreplay":
    os.chdir("slidyreplay")
ok = run("pip install -q -r requirements.txt")
print(f"{'  OK' if ok else 'FAIL'} pip install")
if ok:
    run("apt-get install -y -qq software-properties-common")
    run("add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg6")
    ok = run("apt-get update -qq") and run("apt-get install -y -qq ffmpeg")
    print(f"{'  OK' if ok else 'FAIL'} ffmpeg install")
    if ok:
        run("ffmpeg -version 2>&1 | head -2", show=True)
import torch
has_cuda = torch.cuda.is_available()
print(f"{'  OK' if has_cuda else 'FAIL'} CUDA" + (f" ({torch.cuda.get_device_name(0)})" if has_cuda else " NOT AVAILABLE"))

In [ ]:
#@title ## Generate Replay
import ipywidgets as widgets
from IPython.display import display, Video, clear_output
import subprocess, sys, re, os
from concurrent.futures import ThreadPoolExecutor

mode = widgets.Dropdown(options=['URL Mode', 'Custom Mode'], value='URL Mode', description='Mode:', style={'description_width': 'initial'})
url = widgets.Text(value='https://slidysim.github.io/replay?r=eNqVVNmOm0gU%2FRWLV6wGzN7qZIRN2wY3GLPY2EkUFVRhMPtuSPLvg7sz0czjqB6q7jl3qSrpnC8%2FsD6CTYg9c3MsRNE1bN6PCQIQVV4OKmgPBcKesSZKETbH%2FDxrqjypJ0TL2%2FoBXUGKtBw%2BkqwGZHCqmVAY1UUCht%2FV%2FyKyKX0dJQ2qJhwWIUyzCQXdNUEZ9kzNPyY986IozrE079A0a0HRJElOVPEIOG4hfKTVDUiLqYinBZ6hKXaxoKYGdZ506DsEDfgOOhAlwEumhk3VojnWRRDlSZTF0%2FCwmfo9E0Tf909D3jath578PCV60PjhX92n%2B4nbHOhR3bnxdMOoPiEPew5AUqNfcwypdz15bS1WPmvminjlLBk3DfNkcaTjWO2Wp3EdQfnsWbhjmrKqbrkap%2FVgxBErGnwe8TCkWXGrBfa433c82fpBNZJuk3AG0a6lNDBo1icYWXUIb9m7Wdset%2F7uZhBLjuVXYmhsETEkiOEZhItjd%2FMJAwbCG0NwKm6wrhiwAkVBOuC6c4NfXK7LcPUo8jyOE2JZoDyFarZzvUQs0KYqScBmQLt1byAiZFhnCboAWkDOjhEaf%2BdtrQCeFkefpc2bdxI8ao%2BjCpYMHTS6Ww%2FCeekbeutaqnlXVkq6l%2FqzQNAHaRt6tHVg76wqFYdYibJV7L4tolaOd%2FvDyOo6s2dOr7q0Xh2LzXK1lx0nPFpHLSr3yKs3blHWC58qcS6xu6o0bPfyRku1nNUsujmlGZjgMKYjJEPL5vXyAgTd66E1OscNeXh9UzbBWjO1XI03TXHkpXKhl45b3948LVywMrysVktqMJUIX68ZMlzuap%2FvPHkAJgfrUmPs27W%2FUxW6ddVg24lIjZZrmMF%2BXx79Klfd7VFje8%2FRHVko%2B4Lt7rUEs2J%2FWpJr7nVnZ%2BXmoMk7fOPUw8m%2BX3EjeCtiF17129omr5f8oPV%2BWErWIdbl1t45pKacR3kV9%2FGViiRDkfxzdo2vN1Jpr5vidXCE%2BKwV5Vm1Pak%2BKa8MM%2BaiEql92nMbmeIMt9ABbYB4bETVovxRryWVVgYbh2axFC%2BWLUvVWtlaq9Yu%2FbYI4FnZrnjDF%2BVqI1%2BFW06im6QyaHB4SQOa75%2BrThGtojycW8odFS3PL0TWq50xMHa83tEqTBxjn66nP1D2K%2FkoL9ISmiGZ1tpQjdtdq5ZFEVHCPRZg0IsL0a0PdYXNf0xysqu2bhD8UOav%2BYesuWlhL3UBspmfgLr%2B9BUDSRGCKYr8%2BKHkr9jnlyi9%2FqFRmt%2Bir9isrvwpilJwRTXxj5wjP58okDQTdc5be8Jm7xYw%2B%2BMMj37%2FmXcyjeUEfjmZ32YvxIP6bD0cZeYNsw%2B%2FevGqz%2FZkPzNjOfs5e%2FfBaV%2BQC%2FaJXDyR7O%2BqyTT%2Bn8W8vxSbP%2F7j29%2F7Pdq8', description='URL:', layout=widgets.Layout(width='100%'))
solution = widgets.Text(value='RU2RDLULD2RU2RDL3DRULUR3D2L2U2L', description='Solution:')
size = widgets.Dropdown(options=['2x2','3x3','4x4','5x5','6x6','7x7','8x8','9x9','10x10','11x11','12x12','13x13','14x14','15x15','16x16','17x17','18x18','19x19','20x20'], value='4x4', description='Size:')
tps = widgets.FloatText(value=26.96, description='TPS:')
scramble = widgets.Text(value='1 2 3 4/9 10 0 7/11 8 15 6/5 14 13 12', description='Scramble:')
movetimes_csv = widgets.Text(value='0,60,98,120,158,203,256,308,330,359,390,444,471,496,547,586,637,660,719,766,810,832,861,976,976,976,998,1020,1043,1095,1110,1178,1224', description='Movetimes:', layout=widgets.Layout(width='100%'))
fps = widgets.IntSlider(value=60, min=5, max=240, step=5, description='FPS:')
quality = widgets.Dropdown(options=['1.0', '2.0'], value='1.0', description='Quality:')
compression = widgets.IntSlider(value=18, min=10, max=40, step=1, description='Compression (CRF):')
output = widgets.Text(value='replay.mp4', description='Output:')

custom_fields = widgets.VBox([solution, size, tps, scramble, movetimes_csv])
def toggle_custom(change):
    custom_fields.layout.display = 'flex' if change.new == 'Custom Mode' else 'none'
    url.layout.display = 'flex' if change.new == 'URL Mode' else 'none'
mode.observe(toggle_custom, names='value')
toggle_custom({'new': mode.value})

btn = widgets.Button(description='Generate', button_style='primary', layout=widgets.Layout(width='200px'))
out = widgets.Output()

def _run(args):
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
    ffmpeg_re = re.compile(r'(ffmpeg version|  built with|  configuration:|\blibav|Input #|Stream mapping:|\[libx264|\[mp4 @|\[out#0|\[h264_nvenc|frame= |Lsize=)')
    def read_stream(stream, is_err):
        for line in iter(stream.readline, ''):
            if not is_err or not ffmpeg_re.search(line):
                sys.stdout.write(line); sys.stdout.flush()
    with ThreadPoolExecutor(max_workers=2) as pool:
        pool.submit(read_stream, p.stdout, False)
        pool.submit(read_stream, p.stderr, True)
    p.wait()
    return p.returncode

def on_generate(b):
    with out:
        clear_output(wait=True)
        q = float(quality.value)
        if mode.value == 'URL Mode':
            ret = _run(['python', 'main.py', '--url', url.value, '--fps', str(fps.value), '--quality', str(q), '--compression', str(compression.value), '-o', output.value])
        else:
            ret = _run(['python', 'main.py', '--solution', solution.value, '--size', size.value, '--tps', str(tps.value), '--time', '1.224', '--fps', str(fps.value), '--quality', str(q), '--compression', str(compression.value), '--scramble', scramble.value, '--movetimes', movetimes_csv.value, '-o', output.value])
        if os.path.exists(output.value):
            size_mb = os.path.getsize(output.value) / 1e6
            print(f"\n{'='*50}")
            print(f"File: {output.value}  ({size_mb:.2f} MB)")
            print(f"{'='*50}")
            display(Video(output.value, embed=True, width=640))

btn.on_click(on_generate)
display(mode, url, custom_fields, widgets.HBox([fps, quality, compression]), output, btn, out)

In [ ]:
#@title ## Download Replay
import os
if os.path.exists(output):
    from google.colab import files
    files.download(output)
else:
    print("No replay file found. Generate one first.")

In [ ]:
#@title ## Run Benchmarks

benchmark_mode = "All" #@param ["All", "Small (CPU+GPU)", "Big (GPU only)"]

if benchmark_mode == "All":
    !python benchmark.py
elif benchmark_mode == "Small (CPU+GPU)":
    !python benchmark.py --small
else:
    !python benchmark.py --big